# Kalshi Iran Prediction Markets: A Deep Dive

We're going to look at how Kalshi — the only CFTC-regulated prediction market exchange in the US — is handling Iran-related betting markets. We'll break down where the money is going, which outcomes bettors are most interested in, and how Kalshi stacks up against its bigger (and unregulated) rival, Polymarket.

Let's get started.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Soma-style: set a clean look right up top
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

## 1. Load and Explore the Data

First things first — let's read in our Kalshi CSV and see what we're working with. Getting a feel for the shape and columns is always step one.

In [ ]:
df = pd.read_csv("kalshi_iran_markets.csv")

print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nColumns:")
for col in df.columns:
    print(f"  - {col}")

In [ ]:
df.head(11)

OK so we have 11 markets total. Each row is a Kalshi market about Iran — things like who the next Supreme Leader will be, whether the Strait of Hormuz gets closed, whether Reza Pahlavi returns, etc.

The key columns for us:
- **Market Title** — what the market is about
- **Total Volume ($)** — how much money has been traded
- **Top Outcome Probability (%)** — the market's current best guess
- **Status** — whether the market is still active

Let's check the data types to make sure volume is numeric.

In [ ]:
print(df.dtypes)
print(f"\nVolume column dtype: {df['Total Volume ($)'].dtype}")

## 2. Total Volume: How Much Money Are We Talking About?

Let's add up all the volume across every Iran-related market on Kalshi and see the big picture number.

In [ ]:
total_volume = df['Total Volume ($)'].sum()

print(f"Total volume across all 11 Kalshi Iran markets: ${total_volume:,.0f}")
print(f"That's roughly ${total_volume / 1_000_000:.1f}M")

So about **$6.6 million** has been traded across all of Kalshi's Iran-related prediction markets. That sounds like a lot, but as we'll see later, it's a tiny fraction of what Polymarket is doing. Let's break it down market by market.

## 3. Market-by-Market Breakdown

Let's look at every single market — its volume, the top predicted outcome, the probability, and the status. We'll sort by volume so the biggest markets are at the top.

In [ ]:
breakdown = df[['Market Title', 'Total Volume ($)', 'Top Outcome',
                'Top Outcome Probability (%)', 'Status']].copy()

breakdown = breakdown.sort_values('Total Volume ($)', ascending=False).reset_index(drop=True)
breakdown.index = breakdown.index + 1  # 1-indexed for readability

# Format volume for display
breakdown['Volume (formatted)'] = breakdown['Total Volume ($)'].apply(
    lambda x: f"${x:,.0f}" if x >= 1000 else f"${x:.0f}"
)

print("All 11 Kalshi Iran Markets (sorted by volume):\n")
for i, row in breakdown.iterrows():
    print(f"{i:2d}. {row['Market Title']}")
    print(f"    Volume: {row['Volume (formatted)']}  |  Top outcome: {row['Top Outcome']} ({row['Top Outcome Probability (%)']}%)  |  {row['Status']}")
    print()

## 4. Top Markets by Volume

Now let's visualize this. Which markets are people actually putting money on? A horizontal bar chart is a nice way to see this — especially since our market titles are long.

In [ ]:
# Sort ascending for horizontal bar chart (so biggest is at top visually)
plot_df = breakdown.sort_values('Total Volume ($)', ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))

colors = ['#4C72B0' if v >= 100_000 else '#A8C8E8' for v in plot_df['Total Volume ($)']]

bars = ax.barh(range(len(plot_df)), plot_df['Total Volume ($)'], color=colors, edgecolor='white')

ax.set_yticks(range(len(plot_df)))

# Shorten labels for readability
short_labels = []
for title in plot_df['Market Title']:
    if len(title) > 50:
        short_labels.append(title[:47] + '...')
    else:
        short_labels.append(title)
ax.set_yticklabels(short_labels, fontsize=10)

# Add value labels on bars
for bar_obj, val in zip(bars, plot_df['Total Volume ($)']):
    if val >= 1_000_000:
        label = f'${val/1_000_000:.1f}M'
    elif val >= 1_000:
        label = f'${val/1_000:.0f}K'
    else:
        label = f'${val:.0f}'
    ax.text(bar_obj.get_width() + 20000, bar_obj.get_y() + bar_obj.get_height()/2,
            label, va='center', fontsize=9, fontweight='bold')

ax.set_xlabel('Total Volume ($)', fontsize=12)
ax.set_title('Kalshi Iran Markets by Trading Volume', fontsize=14, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x/1_000_000:.1f}M' if x >= 1_000_000 else f'${x/1_000:.0f}K'))

sns.despine(left=True)
plt.tight_layout()
plt.show()

The big takeaway: the **Next Supreme Leader** market absolutely dominates at $3.3M — that's about half of all the Iran-related volume on Kalshi. The **Strait of Hormuz** market comes in second at $1.6M. After that there's a big drop-off.

Those top two markets account for roughly 74% of all Iran volume on the platform.

In [ ]:
top2 = breakdown.nlargest(2, 'Total Volume ($)')['Total Volume ($)'].sum()
print(f"Top 2 markets: ${top2:,.0f}")
print(f"Share of total: {top2 / total_volume * 100:.1f}%")

## 5. Kalshi vs. Polymarket: A Side-by-Side Comparison

Now here's where it gets really interesting. Let's bring in Polymarket data and see how these two platforms compare on overlapping Iran topics.

Polymarket is the 800-pound gorilla of prediction markets — it's unregulated and crypto-based, which means it tends to attract way more volume. Let's see just *how much* more.

In [ ]:
poly = pd.read_csv("polymarket_iran_markets.csv")

print(f"Polymarket: {poly.shape[0]} markets")
print(f"Polymarket total Iran volume: ${poly['Total Volume ($)'].sum():,.2f}")
print(f"That's ${poly['Total Volume ($)'].sum() / 1_000_000:.0f}M")

Whoa. Over **$950 million** on Polymarket vs. about **$6.6 million** on Kalshi. But a lot of those Polymarket markets don't have Kalshi equivalents (like specific strike date markets, ceasefire bets, etc.).

Let's build a fair comparison by matching up the categories where both platforms have markets — or where one platform has a market and the other doesn't. This tells us where each platform's bettors are putting their attention.

In [ ]:
# Build comparison of shared/comparable categories
# We manually match these based on topic overlap

# Kalshi volumes (from our data)
kalshi_supreme_leader = df[df['Market Title'].str.contains('Supreme Leader', case=False)]['Total Volume ($)'].sum()
kalshi_hormuz = df[df['Market Title'].str.contains('Strait of Hormuz', case=False)]['Total Volume ($)'].sum()
kalshi_nuclear = df[df['Market Title'].str.contains('nuclear deal', case=False)]['Total Volume ($)'].sum()
kalshi_pahlavi_lead = df[df['Market Title'].str.contains('Pahlavi lead', case=False)]['Total Volume ($)'].sum()
kalshi_pahlavi_visit = df[df['Market Title'].str.contains('Pahlavi visit', case=False)]['Total Volume ($)'].sum()
kalshi_pahlavi_recognized = df[df['Market Title'].str.contains('recognize Reza Pahlavi', case=False)]['Total Volume ($)'].sum()
kalshi_invasion = 0  # Kalshi has no US invasion market

# Polymarket volumes for matching topics
poly_supreme_leader = poly[poly['Market Title'].str.contains('Supreme Leader', case=False)]['Total Volume ($)'].sum()
poly_new_leader = poly[poly['Market Title'].str.contains('New Supreme Leader', case=False)]['Total Volume ($)'].sum()
poly_hormuz = 0  # Polymarket has no Strait of Hormuz market
poly_nuclear = 0  # Polymarket has no nuclear deal market
poly_pahlavi_lead = 0  # No direct equivalent
poly_pahlavi_visit = 0  # No direct equivalent
poly_pahlavi_recognized = poly[poly['Market Title'].str.contains('Pahlavi', case=False)]['Total Volume ($)'].sum()
poly_invasion = poly[poly['Market Title'].str.contains('invade Iran', case=False)]['Total Volume ($)'].sum()

print(f"Kalshi - Next Supreme Leader: ${kalshi_supreme_leader:,.0f}")
print(f"Polymarket - New Supreme Leader (direct match): ${poly_new_leader:,.0f}")
print(f"Polymarket - All 'Supreme Leader' markets combined: ${poly_supreme_leader:,.0f}")
print()
print("Note: Polymarket's Supreme Leader markets are mostly about")
print("Khamenei leaving/being removed — a different angle than Kalshi's")
print("'who will be NEXT' market. We'll use the direct 'New Supreme Leader'")
print(f"market (${poly_new_leader:,.0f}) for the fairest comparison.")

In [ ]:
# Build the comparison table
comparison = pd.DataFrame({
    'Category': [
        'Next Supreme Leader',
        'Strait of Hormuz',
        'Nuclear Deal',
        'Pahlavi: Lead Iran',
        'Pahlavi: Visit Iran',
        'Pahlavi: Recognized',
        'U.S. Invasion'
    ],
    'Kalshi Volume': [
        kalshi_supreme_leader,
        kalshi_hormuz,
        kalshi_nuclear,
        kalshi_pahlavi_lead,
        kalshi_pahlavi_visit,
        kalshi_pahlavi_recognized,
        kalshi_invasion
    ],
    'Polymarket Volume': [
        poly_new_leader,
        poly_hormuz,
        poly_nuclear,
        poly_pahlavi_lead,
        poly_pahlavi_visit,
        poly_pahlavi_recognized,
        poly_invasion
    ]
})

# Format for display
def format_vol(x):
    if x == 0:
        return '$0'
    elif x >= 1_000_000:
        return f'${x/1_000_000:.1f}M'
    elif x >= 1_000:
        return f'${x/1_000:.0f}K'
    else:
        return f'${x:,.0f}'

comparison['Kalshi'] = comparison['Kalshi Volume'].apply(format_vol)
comparison['Polymarket'] = comparison['Polymarket Volume'].apply(format_vol)
comparison['Winner'] = comparison.apply(
    lambda row: 'Kalshi' if row['Kalshi Volume'] > row['Polymarket Volume']
    else ('Polymarket' if row['Polymarket Volume'] > row['Kalshi Volume'] else 'Tie'),
    axis=1
)

print("Kalshi vs. Polymarket: Side-by-Side on Comparable Iran Categories")
print("=" * 75)
print(f"{'Category':<25} {'Kalshi':>12} {'Polymarket':>12} {'Leader':>10}")
print("-" * 75)
for _, row in comparison.iterrows():
    print(f"{row['Category']:<25} {row['Kalshi']:>12} {row['Polymarket']:>12} {row['Winner']:>10}")

print("-" * 75)
kalshi_comp_total = comparison['Kalshi Volume'].sum()
poly_comp_total = comparison['Polymarket Volume'].sum()
print(f"{'TOTAL':<25} {format_vol(kalshi_comp_total):>12} {format_vol(poly_comp_total):>12}")

Now let's look at the overall volume gap between the two platforms across *all* their Iran markets — not just the ones that overlap.

In [ ]:
kalshi_total = df['Total Volume ($)'].sum()
poly_total = poly['Total Volume ($)'].sum()

volume_gap = poly_total / kalshi_total

print(f"Kalshi total Iran volume:      ${kalshi_total:>15,.0f}  ({df.shape[0]} markets)")
print(f"Polymarket total Iran volume:  ${poly_total:>15,.0f}  ({poly.shape[0]} markets)")
print(f"")
print(f"Polymarket has {volume_gap:.0f}x more volume than Kalshi on Iran markets.")
print(f"")
print(f"Put another way: for every $1 traded on Kalshi,")
print(f"about ${volume_gap:.0f} is traded on Polymarket.")

In [ ]:
# Visualize the head-to-head comparison
fig, ax = plt.subplots(figsize=(12, 7))

categories = comparison['Category']
x = range(len(categories))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], comparison['Kalshi Volume'], width,
               label='Kalshi', color='#4C72B0', edgecolor='white')
bars2 = ax.bar([i + width/2 for i in x], comparison['Polymarket Volume'], width,
               label='Polymarket', color='#DD8452', edgecolor='white')

ax.set_ylabel('Volume ($)', fontsize=12)
ax.set_title('Kalshi vs. Polymarket: Volume on Comparable Iran Categories',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=30, ha='right', fontsize=10)
ax.legend(fontsize=11)

ax.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, p: f'${x/1_000_000:.1f}M' if x >= 1_000_000 else f'${x/1_000:.0f}K' if x >= 1_000 else f'${x:.0f}'
))

# Add value labels
for bar_obj in bars1:
    val = bar_obj.get_height()
    if val > 0:
        ax.text(bar_obj.get_x() + bar_obj.get_width()/2, val + 15000,
                format_vol(val), ha='center', va='bottom', fontsize=8, fontweight='bold', color='#4C72B0')

for bar_obj in bars2:
    val = bar_obj.get_height()
    if val > 0:
        ax.text(bar_obj.get_x() + bar_obj.get_width()/2, val + 15000,
                format_vol(val), ha='center', va='bottom', fontsize=8, fontweight='bold', color='#DD8452')

sns.despine()
plt.tight_layout()
plt.show()

## 6. Where Kalshi Actually Wins

OK so Polymarket crushes Kalshi overall on volume. That's not news — Polymarket is bigger on basically everything. But there's a more interesting question: **are there categories where Kalshi actually has more action than Polymarket?**

Turns out, yes. And the pattern is fascinating.

In [ ]:
kalshi_wins = comparison[comparison['Kalshi Volume'] > comparison['Polymarket Volume']].copy()

print(f"Categories where Kalshi has MORE volume than Polymarket: {len(kalshi_wins)}")
print("=" * 70)

for _, row in kalshi_wins.iterrows():
    diff = row['Kalshi Volume'] - row['Polymarket Volume']
    if row['Polymarket Volume'] > 0:
        ratio = row['Kalshi Volume'] / row['Polymarket Volume']
        print(f"\n  {row['Category']}")
        print(f"    Kalshi: {row['Kalshi']}  vs  Polymarket: {row['Polymarket']}")
        print(f"    Kalshi has {ratio:.1f}x more volume (${diff:,.0f} more)")
    else:
        print(f"\n  {row['Category']}")
        print(f"    Kalshi: {row['Kalshi']}  vs  Polymarket: {row['Polymarket']}")
        print(f"    Only on Kalshi — Polymarket has no equivalent market")

In [ ]:
# Let's make a focused chart on just the Kalshi-wins categories

fig, ax = plt.subplots(figsize=(10, 5))

cats = kalshi_wins['Category']
x = range(len(cats))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], kalshi_wins['Kalshi Volume'], width,
               label='Kalshi', color='#4C72B0', edgecolor='white')
bars2 = ax.bar([i + width/2 for i in x], kalshi_wins['Polymarket Volume'], width,
               label='Polymarket', color='#DD8452', edgecolor='white', alpha=0.7)

ax.set_ylabel('Volume ($)', fontsize=12)
ax.set_title('Categories Where Kalshi Beats Polymarket on Volume',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(cats, rotation=20, ha='right', fontsize=10)
ax.legend(fontsize=11)

ax.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, p: f'${x/1_000_000:.1f}M' if x >= 1_000_000 else f'${x/1_000:.0f}K' if x >= 1_000 else f'${x:.0f}'
))

for bar_obj in bars1:
    val = bar_obj.get_height()
    if val > 0:
        ax.text(bar_obj.get_x() + bar_obj.get_width()/2, val + 10000,
                format_vol(val), ha='center', va='bottom', fontsize=9, fontweight='bold', color='#4C72B0')

for bar_obj in bars2:
    val = bar_obj.get_height()
    if val > 0:
        ax.text(bar_obj.get_x() + bar_obj.get_width()/2, val + 10000,
                format_vol(val), ha='center', va='bottom', fontsize=9, fontweight='bold', color='#DD8452')

sns.despine()
plt.tight_layout()
plt.show()

This is genuinely interesting. Kalshi's edge is concentrated in two areas:

1. **The Next Supreme Leader** -- Kalshi's market asks *who* will be next (Arafi? Khomeini? Mojtaba?), while Polymarket's markets focus more on *whether/when* Khamenei leaves. Kalshi's framing attracted $3.3M vs. Polymarket's $1.1M on the successor question specifically.

2. **Pahlavi and Iran-specific politics** -- Kalshi has a whole cluster of markets about Reza Pahlavi (visiting Iran, leading Iran, being recognized by the US). These are nuanced, Iran-watcher type bets. Polymarket only has one Pahlavi market (the recognition one), and Kalshi beats it there too.

3. **Unique markets** -- Kalshi is the only place to bet on the Strait of Hormuz closure ($1.6M) or a US-Iran nuclear deal ($583K). Polymarket doesn't offer either.

## 7. Key Findings

Here's what we learned from digging into Kalshi's Iran prediction markets:

**The big numbers:**
- Kalshi has **11 Iran-related markets** with a combined **$6.6M** in trading volume
- The **Next Supreme Leader** market dominates at **$3.3M** -- roughly half of all Kalshi Iran volume
- The **Strait of Hormuz** market is second at **$1.6M**
- Those top two markets account for about **74%** of all Iran volume on the platform

**The Polymarket comparison:**
- Polymarket has **~143x** more total volume on Iran markets (~$950M vs ~$6.6M)
- But Kalshi actually **wins** on several specific categories:
  - Next Supreme Leader: Kalshi $3.3M vs Polymarket $1.1M
  - Strait of Hormuz: Kalshi $1.6M vs Polymarket $0
  - Nuclear Deal: Kalshi $583K vs Polymarket $0
  - Pahlavi: Lead Iran: Kalshi $462K vs Polymarket $0
  - Pahlavi: Visit Iran: Kalshi $365K vs Polymarket $0
  - Pahlavi: Recognized: Kalshi $340K vs Polymarket $284K
- Polymarket's edge: U.S. Invasion ($266K vs $0 on Kalshi)

**The takeaway:**
Kalshi is a regulated minnow swimming in Polymarket's ocean. But it has found its niche in *political specificity* -- questions about Iran's internal politics (who leads next, what happens with Pahlavi, the Strait of Hormuz) where Kalshi's market design attracts more informed, targeted betting. Polymarket dominates the big, dramatic questions (will there be strikes? will the regime fall?), but Kalshi owns the nuanced ones.